<a href="https://colab.research.google.com/github/Shanmukh-dev/Custom-GPT/blob/main/CustomGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch import nn
import math
device = "cuda" if torch.cuda.is_available() else "cpu"
device

# Transformer

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, n_heads):
    super().__init__()

    assert d_model % n_heads == 0

    self.d_model = d_model
    self.n_heads = n_heads
    self.head_dim = d_model // n_heads

    self.query = nn.Linear(in_features=d_model, out_features=d_model).to(device)
    self.key = nn.Linear(in_features=d_model, out_features=d_model).to(device)
    self.value = nn.Linear(in_features=d_model, out_features=d_model).to(device)

    self.out_proj = nn.Linear(d_model, d_model).to(device)

  def forward(self, X):
    X = X.to(device)
    B, T, C = X.shape
    Q, K, V = self.query(X), self.key(X), self.value(X)

    Q = Q.view(B, T, self.n_heads, self.head_dim).to(device)
    K = K.view(B, T, self.n_heads, self.head_dim).to(device)
    V = V.view(B, T, self.n_heads, self.head_dim).to(device)

    Q = Q.transpose(1, 2)
    K = K.transpose(1, 2)
    V = V.transpose(1, 2)

    scores = Q @ K.transpose(-1, -2)
    scores = scores / math.sqrt(self.head_dim)


    causal_mask = torch.tril(torch.ones(T, T)).to(device)
    scores = scores.masked_fill(causal_mask == 0, float("-inf")).to(device)

    attn_weights = torch.softmax(scores, dim = -1).to(device)

    attn = attn_weights @ V


    attn = attn.transpose(1, 2)

    attn = attn.contiguous().view(B, T, self.d_model).to(device)

    attn = self.out_proj(attn)
    return attn




class MLP(nn.Module):
  def __init__(self, input_dimensions, hidden_layers, output_dimensions) -> None:
    super().__init__()

    self.mpl_layer = nn.Sequential(
        nn.Linear(in_features=input_dimensions, out_features=hidden_layers).to(device),
        nn.GELU().to(device),
        nn.Linear(in_features=hidden_layers, out_features=output_dimensions).to(device)
    ).to(device)


  def forward(self, X):
    X = X.to(device)
    return self.mpl_layer(X)


class TransformerBlock(nn.Module):
  def __init__(self, d_model, n_heads, hidden_layers):
    super().__init__()

    self.multi_head_attention = MultiHeadAttention(d_model, n_heads).to(device)
    self.mlp = MLP(d_model, hidden_layers, d_model).to(device)

    self.ln1 = nn.LayerNorm(d_model).to(device)
    self.ln2 = nn.LayerNorm(d_model).to(device)


  def forward(self, X):
    X = X.to(device)

    normalized_X = self.ln1(X)
    attention_weights = self.multi_head_attention(normalized_X)
    attention_weights = X + attention_weights


    normalized_attention_weights = self.ln2(attention_weights).to(device)
    mlp_output = self.mlp(normalized_attention_weights)

    output = attention_weights + mlp_output

    return output




# GPT Modle


In [ ]:
class CustomGPT(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.token_embeddings = nn.Embedding(config.vocab_size, config.d_model)

    self.positional_embeddings = nn.Embedding(config.block_size, config.d_model)


    self.transformer_blocks = nn.ModuleList(
        [
            TransformerBlock(config.d_model, config.n_heads, config.hidden_layers)
            for _ in range(config.n_layers)

        ]
    )


    self.layer_norm = nn.LayerNorm(config.d_model)
    self.lm_head = nn.Linear(config.d_model, config.vocab_size)
    self.config = config


  def forward(self, idx):
    B, T = idx.shape

    tok_embd = self.token_embeddings(idx)
    pos = torch.arange(T, device=device)

    pos_embd = self.positional_embeddings(pos)

    X = tok_embd + pos_embd

    for block in self.transformer_blocks:
      X = block(X)

    X = self.layer_norm(X)

    logits = self.lm_head(X)

    return logits


  @torch.no_grad()
  def generate(self, idx, max_new_tokens=500):
    for _ in range(max_new_tokens):
      idx_cont = idx[:, -self.config.block_size:]

      logits = self(idx_cont)

      logits = logits[:, -1, :]

      probs = torch.softmax(logits, dim=-1)
      new_tokens = torch.multinomial(probs, num_samples=1)
      idx = torch.cat((idx, new_tokens), dim=1)

    return idx


# The dataset and confg

In [ ]:
from datasets import load_dataset

ds = load_dataset("PleIAs/SYNTH", split="train", streaming = True)

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

tokens = []
target_tokens = 10_000_000

for sample in ds:
    text = (
        sample["query"]+
        sample["query_seed_text"]+
        sample["synthetic_reasoning"]+
        sample["synthetic_answer"]
    )

    ids = tokenizer.encode(text)

    tokens.extend(ids)


    if len(tokens) >= target_tokens:
        break


print(len(tokens))

In [ ]:
from torch.utils.data import Dataset, DataLoader

data = torch.tensor(tokens, dtype=torch.long, device=device)
print(len(data))


train_split = int(0.9*len(data))
train_data = data[:train_split]
test_data = data[train_split:]
print("Train data length:", len(train_data))
print("Test data length:", len(test_data))

In [ ]:
class GPTConfig:
  vocab_size = tokenizer.n_vocab
  block_size = 256

  d_model = 256
  hidden_layers = 1024
  n_heads = 4
  n_layers = 6

In [ ]:
class SynthDataset(Dataset):
  def __init__(self, data, block_size):
    self.data = data
    self.block_size = block_size

  def __len__(self):
    return len(self.data) - self.block_size

  def __getitem__(self, idx):
    x = data[idx:idx+self.block_size]
    y = data[idx+1:idx+self.block_size+1]
    return x, y

In [ ]:
train_dataset = SynthDataset(train_data, GPTConfig.block_size)
test_dataset = SynthDataset(test_data, GPTConfig.block_size)

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [ ]:
tokenizer.n_vocab

# The training loop

In [ ]:
model = CustomGPT(GPTConfig).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr = 3e-4)

In [ ]:
from tqdm.auto import tqdm
steps = 50000

torch.manual_seed(42)
train_iter = iter(train_dataloader)
for step in tqdm(range(1, steps+1)):
  try:
    X_train, y_train = next(train_iter)
  except StopIteration:
    train_iter = iter(train_dataloader)
    X_train, y_train = next(train_iter)

  X_train, y_train = X_train.to(device), y_train.to(device)

  model.train()
  logits = model(X_train)
  optimizer.zero_grad()

  train_loss = loss_fn(logits.view(-1, logits.size(-1)), y_train.view(-1))

  train_loss.backward()
  optimizer.step()

  if step % 10000 == 0:
    model.eval()
    test_loss = 0
    with torch.inference_mode():

      for idx, (X_test, y_test) in tqdm(enumerate(test_dataloader)):
        if idx == 10:
          break
        X_test, y_test = X_test.to(device), y_test.to(device)
        test_logits = model(X_test)

        curr_loss = loss_fn(test_logits.view(-1, test_logits.size(-1)), y_test.view(-1))

        test_loss += curr_loss.item()


    test_loss = test_loss / 20

    print(f"Step: {step} | Training loss: {train_loss} | Testing loss: {test_loss}")





In [ ]:
torch.save(model.state_dict(), "custom_gpt.pth")